# Basic RAG Pipeline on the OrchestrAI Synthetic Enterprise Corpus

This notebook demonstrates a basic Retrieval-Augmented Generation (RAG) pipeline using Haystack and a synthetic internal-document corpus created for the thesis.

**Outputs and IDs:** Results are logged as `implementation="basic"` with scenario IDs `b1`–`b5` into `data/results/experimental/hpc_results.csv` (via `results_logger.py`). Each run clears only the previous `basic` rows before saving fresh outputs. See the repository `README.md` for evaluation and project layout.

The workflow is:
1. Load DOCX files from the OrchestrAI dataset
2. Convert them into Haystack Documents
3. Split and embed the documents
4. Store embeddings in an in-memory document store
5. Retrieve relevant chunks for a user query
6. Generate a grounded answer using an LLM

## Environment requirements

This notebook assumes:
- Python **3.12** (recommended) and project dependencies from `requirements.txt`
- vLLM services are running and reachable from this HPC job/session
- `/nvme/scratch/cy26nk2/llm.env` and `/nvme/scratch/cy26nk2/embd.env` exist with valid API keys, URLs, and model names
- The OrchestrAI corpus lives under **`data/dummy_data/`** (loaded via `PROJECT_ROOT / "data" / "dummy_data"` in code; works when the notebook kernel cwd is `notebooks/` or the repo root)
- The first code cell prepends **`src/`** to `sys.path` so `results_logger` imports resolve
- Dense retrieval uses **`top_k=10`** chunks passed to the LLM (aligned with the Milvus notebooks for a fair comparison)

### Import packages

In [1]:
import sys
from pathlib import Path

# Make the shared helpers importable from the notebook.
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "notebooks-hpc"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import time
from results_logger import save_result_row, clear_results_for_implementation

RESULTS_CSV = PROJECT_ROOT / "data" / "results" / "experimental" / "hpc_results.csv"
# Start this implementation's results from a clean slate.
clear_results_for_implementation("basic", results_csv=RESULTS_CSV)

from haystack import Pipeline
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.embedders import OpenAIDocumentEmbedder, OpenAITextEmbedder
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.utils import Secret
from dotenv import dotenv_values
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever
from haystack.dataclasses import ChatMessage
from haystack.components.builders import ChatPromptBuilder
from haystack.components.converters import DOCXToDocument
from haystack.components.preprocessors import DocumentSplitter
import warnings
warnings.filterwarnings("ignore")
print("Imports work")

Cleared existing rows for implementation='basic'
Imports work


In [2]:
llm_config = dotenv_values('/nvme/scratch/cy26nk2/llm.env')
embd_config = dotenv_values('/nvme/scratch/cy26nk2/embd.env')

print("Loaded LLM model:", llm_config.get("MODEL_NAME", "<missing>"))
print("Loaded embedder model:", embd_config.get("MODEL_NAME", "<missing>"))

Loaded LLM model: meta-llama/Llama-3.1-8B-Instruct
Loaded embedder model: BAAI/bge-m3


### Convert data to Haystack Documents

Haystack uses these abstraction called *Documents*. They can hold text, tables, and binary data.

They have the following unique features:

- Unique ID for each document.
- Multiple content types are supported.
- Custom metadata and scoring for advanced document management.
- Optional embeddings for AI-based applications

**Example:**

```python
@dataclass
class Document(metaclass=_BackwardCompatible):
    id: str = field(default="")
    content: Optional[str] = field(default=None)
    blob: Optional[ByteStream] = field(default=None)
    meta: Dict[str, Any] = field(default_factory=dict)
    score: Optional[float] = field(default=None)
    embedding: Optional[List[float]] = field(default=None)
    sparse_embedding: Optional[SparseEmbedding] = field(default=None)
```

---

To convert our files do Haystack Documents, we need to use a *converter*. In our case, since we only have .docx documents, we can go ahead and use Haystack's DOCXToDocument converter.

In [3]:
DOCUMENTS_DIR = PROJECT_ROOT / "data" / "dummy_data"
FILES = list(DOCUMENTS_DIR.rglob("*.docx"))
converter = DOCXToDocument()

docs = []
for file in FILES:
    result = converter.run(sources=[file.resolve()])
    converted_docs = result["documents"]

    for doc in converted_docs:
        if doc.meta is None:
            doc.meta = {}

        doc.meta["file_name"] = file.name
        doc.meta["file_path"] = str(file.resolve())

        # parent folder under dummy_data becomes the department
        try:
            doc.meta["department"] = file.parent.relative_to(DOCUMENTS_DIR).as_posix()
        except ValueError:
            doc.meta["department"] = file.parent.name

    docs.extend(converted_docs)

print(f"Loaded {len(FILES)} DOCX files.")
print(f"Created {len(docs)} Haystack documents.")

Loaded 54 DOCX files.
Created 54 Haystack documents.


#### Inspect a sample document from our database

#### Content

In [4]:
print(docs[0].content)

ORCHESTRAI INTERNAL KNOWLEDGE BASE
Webinar Launch Brief
Document ID: VAL-SALES-002   Department: Sales & Solutions   Confidentiality: Internal Use Only   
Sales / Solutions Validation Document
Campaign: “Scaling Warehouse Orchestration with OrchestrAI Edge Gateways”
Launch summary
Field,Value
Target audience,"Operations leaders, warehouse IT managers, solution architects"
Primary objective,Generate qualified meetings for multi-site orchestration opportunities
Primary CTA,Book an operational assessment
Follow-up owner,Regional BDR team
Success metric,Qualified meeting conversion rate > 12%
Messaging guardrails
• Use approved claims only. Do not promise unsupported roadmap features during promotion or live Q and A.
• All customer examples must be anonymized unless written approval exists.
• Pricing questions should route to the assigned account team rather than being answered in the webinar chat.
Post-event actions
Action,Owner,Due
Upload attendance and engagement signals,Marketing Ops,W

#### Metadata

In [5]:
print(docs[0].meta)

{'file_path': '/nvme/h/cy26nk2/Thesis/data/dummy_data/Sales/VAL-SALES-002_Webinar_Launch_Brief_refreshed.docx', 'docx': {'author': 'python-docx', 'category': '', 'comments': 'generated by python-docx', 'content_status': '', 'created': '2013-12-23T23:15:00+00:00', 'identifier': '', 'keywords': '', 'language': '', 'last_modified_by': '', 'last_printed': None, 'modified': '2013-12-23T23:15:00+00:00', 'revision': 1, 'subject': '', 'title': '', 'version': ''}, 'file_name': 'VAL-SALES-002_Webinar_Launch_Brief_refreshed.docx', 'department': 'Sales'}


## Indexing Documents and performing RAG

### Setup Indexing components

To be able to use our documents we need to perform 2 things:

1. Turn them into embeddings with an *embedder*.
2. Store them in a Haystack *Document Store* so they can be accessed later on.

For our simple use-case, we use a vLLM-served document embedder (OpenAI-compatible endpoint) to extract embeddings, and then we store them in an InMemoryDocumentStore. This means the indexed embeddings are kept in RAM for the current session.

In [6]:
document_store = InMemoryDocumentStore()
doc_embedder = OpenAIDocumentEmbedder(
    api_key=Secret.from_token(embd_config["VLLM_API_KEY"]),
    api_base_url=f"{embd_config['VLLM_EMBD_URL']}/v1",
    model=embd_config["MODEL_NAME"],
)

### Extract embeddings and store to Document Store

Go ahead and run the cell below to begin calculating the embeddings.

In [7]:
indexing_start = time.perf_counter()

splitter = DocumentSplitter(
    split_by="word",
    split_length=120,
    split_overlap=20
)

split_docs = splitter.run(documents=docs)["documents"]

print(f"Original docs: {len(docs)}")
print(f"Split docs: {len(split_docs)}")

docs_with_embeddings = doc_embedder.run(split_docs)
document_store.write_documents(docs_with_embeddings["documents"])

indexing_time_s = round(time.perf_counter() - indexing_start, 4)

print(f"Stored {len(docs_with_embeddings['documents'])} split documents.")
print(f"Indexing time (s): {indexing_time_s}")

Original docs: 54
Split docs: 279


Calculating embeddings: 9it [00:01,  5.31it/s]

Stored 279 split documents.
Indexing time (s): 1.7365


Stored 279 split documents.
Indexing time (s): 1.8499


### Initialize RAG

#### Prompt Template for user

This prompt template will be used by our LLM to generate a response based on our Query.

Specifically, the LLM will read this text from top to bottom:

- It will read the task, which is to respond to the user's query using the **provided context**.
- It will then read some **General Guidelines**.
- Then it will read the **provided context**.
- And finally it will read the **user's query**.

You can see that we pass the **context** and **user's query** through this template. This is why we use a prompt builder later on. This component constructs prompts dynamically by processing chat messages.

Specifically, the *ChatPromptBuilder* component creates prompts using static or dynamic templates written in Jinja2 syntax, by processing a list of chat messages. The templates contain placeholders like {{ variable }} that are filled with values provided during runtime. You can use it for static prompts set at initialization or change the templates and variables dynamically while running.

In [8]:
template = [
    ChatMessage.from_user(
        """
Respond to the User Query using only the provided Context.

General Guidelines:
- Use only the provided context.
- If the answer is not found in the context, say so clearly.
- Do not make up facts, document sections, pages, or citations.
- If the answer comes from multiple documents, cite all relevant documents.
- Respond in the same language as the user’s query.
- Do not use emojis.
- Be professional and concise.
- Do not write a conclusion or follow-up unless the user asks for it.
- If the context contains a direct policy rule, exception, SLA, or escalation instruction, state it directly and do not hedge.
- Prefer the most specific procedural source over broader handbook summaries when both are present.

Citation Rules:
- For every important claim, cite the source document filename.
- If available in the retrieved context, also mention the document ID or department.
- Do not mention source chapter, section, or page unless they are explicitly present in the retrieved context.

Context:
{% for document in documents %}
[Source: {{ document.meta.get("file_name", document.meta.get("file_path", "unknown source")) }}]
{{ document.content }}

{% endfor %}

Question: {{question}}
Answer:
"""
    )
]

#### RAG Components

For the actual RAG pipeline we have the following components:

- The *text_embedder* which takes the user's query and turns it into embeddings
- The *retriever* which retrieves the relevant documents
- The *chat_generator* which is our LLM
- The *promt_builder* which was explained above.

In [9]:
text_embedder = OpenAITextEmbedder(
    api_key=Secret.from_token(embd_config["VLLM_API_KEY"]),
    api_base_url=f"{embd_config['VLLM_EMBD_URL']}/v1",
    model=embd_config["MODEL_NAME"],
)

retriever = InMemoryEmbeddingRetriever(document_store=document_store)

chat_generator = OpenAIChatGenerator(
    api_key=Secret.from_token(llm_config["VLLM_API_KEY"]),
    api_base_url=f"{llm_config['VLLM_LLM_URL']}/v1",
    model=llm_config["MODEL_NAME"],
    generation_kwargs={
        "temperature": 0.0,
        "top_p": 1.0,
        "seed": 42,
    }
)

prompt_builder = ChatPromptBuilder(
    template=template,
    required_variables=["question", "documents"]
)

# Initialize RAG pipeline
basic_rag_pipeline = Pipeline()

basic_rag_pipeline.add_component("text_embedder", text_embedder)
basic_rag_pipeline.add_component("retriever", retriever)
basic_rag_pipeline.add_component("prompt_builder", prompt_builder)
basic_rag_pipeline.add_component("llm", chat_generator)

# Connect the input/output of each component
basic_rag_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
basic_rag_pipeline.connect("retriever.documents", "prompt_builder.documents")
basic_rag_pipeline.connect("prompt_builder.prompt", "llm.messages")

🚅 Components
  - text_embedder: OpenAITextEmbedder
  - retriever: InMemoryEmbeddingRetriever
  - prompt_builder: ChatPromptBuilder
  - llm: OpenAIChatGenerator
🛤️ Connections
  - text_embedder.embedding -> retriever.query_embedding (list[float])
  - retriever.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> llm.messages (list[ChatMessage])

We connect each component by defining the inputs and outputs.

For example:

- The *text_embedder* will take the user's query as an input and output *embeddings*. These embeddings will be the input of the *retriever*. The *retriever* will take that input as *query_embedding* and output a list of *documents* that are similar to that query.
- Then, the *retriever* outputs the documents we mentioned, and pass them into our *prompt_builder*.
- Finally, the *prompt_builder* will output the prompt and send it as an input to the *llm*.

#### Perform RAG on our data

Feel free to change the question to something else.

Our documents contain information about the following topics:

- Logistics and field fulfillment operations
- Engineering platform, edge systems, releases, and incidents
- IT workplace support, device provisioning, MFA, VPN, and service desk workflows
- Cybersecurity access, compliance, phishing, patching, retention, and third-party access
- Product and program management, launch readiness, release planning, and change control
- Customer Success and Support, onboarding, severity levels, escalations, and renewal risk
- Sales and Solutions, lead routing, CRM governance, campaigns, and discount exceptions
- HR and People Operations, onboarding, leave policy, recognition, desk moves, and room usage
- Finance and Procurement, purchase orders, reimbursements, vendor onboarding, close reminders, and emergency procurement exceptions

Feel free to ask anything relating to these topics.

Suggested prompts:

1. "What is the first-response SLA for a Sev-1 support incident?"
2. "What steps must be completed before a new employee is considered ready for day one at OrchestrAI?"
3. "A new engineer starts on Monday, but their laptop has not arrived and their required access is still pending. Which teams are involved and what should happen next?"
4. "Can OrchestrAI make an urgent hardware purchase without following the normal PO workflow if a warehouse outage is at risk?"
5. "A customer renewal is at risk because support cases are recurring, a requested feature is still unavailable, and replacement hardware is delayed. What should OrchestrAI do next, and which teams must be involved?"

In [11]:
def run_basic_query(question):
    print("=" * 80)
    print("QUESTION:")
    print(question)
    print("=" * 80)

    retrieval_start = time.perf_counter()

    q_emb = text_embedder.run(text=question)["embedding"]
    retr_out = retriever.run(query_embedding=q_emb, top_k=10)
    top_docs = retr_out["documents"]

    retrieval_time_s = round(time.perf_counter() - retrieval_start, 4)

    print(f"\nRETRIEVED DOCUMENTS: {len(top_docs)}")
    print(f"RETRIEVAL TIME (s): {retrieval_time_s}\n")

    for i, d in enumerate(top_docs, 1):
        source = d.meta.get("file_name", d.meta.get("file_path", "unknown source"))
        department = d.meta.get("department", "unknown department")
        snippet = (d.content or "")[:400].replace("\n", " ")
        print(f"[{i}] Source: {source} | Department: {department}")
        print(f"    {snippet}...")
        print()

    generation_start = time.perf_counter()

    result = basic_rag_pipeline.run({
        "text_embedder": {"text": question},
        "retriever": {"top_k": 10},
        "prompt_builder": {"question": question},
    })

    answer = result["llm"]["replies"][0].text
    generation_time_s = round(time.perf_counter() - generation_start, 4)

    print("=" * 80)
    print("FINAL ANSWER:")
    print("=" * 80)
    print(answer)
    print(f"\nGENERATION TIME (s): {generation_time_s}")

    return top_docs, answer, retrieval_time_s, generation_time_s

In [12]:
def run_and_save_basic(question, scenario_id):
    top_docs, answer, retrieval_time_s, generation_time_s = run_basic_query(question)

    save_result_row(
        implementation="basic",
        scenario_id=scenario_id,
        question=question,
        generated_answer=answer,
        top_docs=top_docs,
        retriever_top_k=10,
        reranker_top_k=None,
        indexing_time_s=indexing_time_s,
        retrieval_time_s=retrieval_time_s,
        generation_time_s=generation_time_s,
        results_csv=RESULTS_CSV,
    )

    return top_docs, answer

### Scenario 1 — Basic fact retrieval
This is the clean baseline. It shows whether the system can retrieve one exact operational fact correctly.
- Single-document, single-fact retrieval.
- Tests basic embedding-based retrieval accuracy.
- Expected answer should come from the Customer Success / Support materials.

In [13]:
top_docs_basic_s1, answer_basic_s1 = run_and_save_basic(
    "What is the first-response SLA for a Sev-1 support incident?",
    scenario_id="b1",
)

QUESTION:
What is the first-response SLA for a Sev-1 support incident?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.0295

[1] Source: TEAMQA-CS-001_OrchestrAI_Customer_Success_and_Support_Handbook.docx | Department: Q&A Hanbooks
    are support severities defined? A: Severity is tied to operational impact. Sev-1 means a critical warehouse process cannot run in production. Sev-2 means a major function is degraded with no acceptable workaround. Sev-3 covers localized defects or time-sensitive questions. Sev-4 is low-impact guidance or enhancement clarification. Q: Who owns customer communication during a Sev-1 incident? A: The ...

[2] Source: VAL-ENG-001_Release_Freeze_Notice.docx | Department: Engineering
    topology changes, and authentication-related configuration updates. Allowed during freeze: security hotfixes, Sev-1 incident mitigations, certificate renewals with rollback, and pre-approved customer break/fix patches. Not allowed during freeze: feature releases, non-critical de

### Scenario 2 — Procedural retrieval
This scenario is more complex because the answer is not a single fact, but a short operational process.
- Procedure and checklist retrieval.
- Tests whether the system can retrieve and summarize structured steps clearly.
- Expected answer should come mainly from the HR / People Operations materials.

In [14]:
top_docs_basic_s2, answer_basic_s2 = run_and_save_basic(
    'What steps must be completed before a new employee is considered ready for day one at OrchestrAI?',
    scenario_id="b2",
)

QUESTION:
What steps must be completed before a new employee is considered ready for day one at OrchestrAI?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.0268

[1] Source: VAL-HR-001_Day_One_Onboarding_Checklist.docx | Department: HR
 Applies to...e,2026-02-24erationsBASE Day-One Onboarding Checklist Document ID: VAL-HR-001   Owner: HR & People Operations   Confidentiality: Internal Use Only    Purpose: This checklist standardizes the minimum tasks that must be complete before an OrchestrAI employee can be considered ready for day-one onboarding. Field,Value

[2] Source: VAL-PROD-001_Launch_Readiness_Checklist.docx | Department: Product
 "Effective Date 2026-03-24","Review Cycle Quarterly","Audience PMO, Product, Engineering, Support, Sales Enab...nfirm whether a customer-facing release may proceed to go-live. "Document ID VAL-PROD-001","Department Product & Program Management","Owner Program Management Office","Version 1.0"

[3] Source: VAL-SEC-005_Third_Party_Access_Approval_Standar

### Scenario 3 — Multi-team operational reasoning
This scenario requires combining information from more than one team to answer a realistic workplace problem.
- Multi-document, cross-team retrieval.
- Tests whether the system can synthesize HR, IT, and Security responsibilities.
- Expected answer should identify both the involved teams and the next actions.

In [15]:
top_docs_basic_s3, answer_basic_s3 = run_and_save_basic(
    'A new engineer starts on Monday, but their laptop has not arrived and their required access is still pending. Which teams are involved and what should happen next?',
    scenario_id="b3",
)

QUESTION:
A new engineer starts on Monday, but their laptop has not arrived and their required access is still pending. Which teams are involved and what should happen next?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.0269

[1] Source: VAL-IT-001_Laptop_Provisioning_FAQ.docx | Department: IT
 Handoff and confirmation,Site IT / ship desk,By start date,Asset receipt and user sign-off  Common Questions Q: How far in advance sho...

[2] Source: TEAMQA-IT-001_OrchestrAI_IT_Workplace_and_Service_Desk_Handbook.docx | Department: Q&A Hanbooks
    are new laptops provisioned? A: Endpoint Engineering stages a standard image, applies FleetPulse policies, encrypts the device, validates baseline software, and associates the asset with the employee record before release. Q: Who approves software installations? A: Standard catalog apps can be self-requested or manager-approved depending on license cost. Non-standard software requires manager appr...

[3] Source: VAL-IT-001_Laptop_Provisioning_FAQ.

### Scenario 4 — Policy + exception handling
This scenario checks whether the system can distinguish the normal rule from the emergency exception path.
- Policy retrieval with exception reasoning.
- Tests whether the system can identify when a standard workflow may be bypassed.
- Expected answer should combine Finance / Procurement rules with operational urgency.

In [16]:
top_docs_basic_s4, answer_basic_s4 = run_and_save_basic(
    'Can OrchestrAI make an urgent hardware purchase without following the normal PO workflow if a warehouse outage is at risk?',
    scenario_id="b4",
)

QUESTION:
Can OrchestrAI make an urgent hardware purchase without following the normal PO workflow if a warehouse outage is at risk?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.0265

[1] Source: TEAMQA-LOG-001_OrchestrAI_Logistics_and_Field_Fulfillment_Handbook.docx | Department: Q&A Hanbooks
    exception decisions were made. Auditability is a first-class requirement because OrchestrAI often needs to reconstruct site hardware history months after deployment. How should warehouse stock counts be corrected after a mismatch? A recount must occur before any system adjustment is posted. If the mismatch remains, the warehouse lead documents the discrepancy, checks recent pick faces, receiving r...

[2] Source: TEAMQA-FIN-001_OrchestrAI_Finance_and_Procurement_Handbook.docx | Department: Q&A Hanbooks
    quarterly budget checkpoints, vendor dependency notes, and cross-functional approval calendars. • High-risk or high-value vendors may require parallel checks from Security, Legal, or the 

### Scenario 5 — Cross-functional enterprise reasoning
This is the most complex scenario because it requires cross-functional reasoning across several business units.
- Multi-hop, multi-document retrieval.
- Tests whether the system can combine support, product, logistics, and commercial risk signals.
- Expected answer should identify the teams involved and propose a coordinated next-step plan.

In [17]:
top_docs_basic_s5, answer_basic_s5 = run_and_save_basic(
    'A customer renewal is at risk because support cases are recurring, a requested feature is still unavailable, and replacement hardware is delayed. What should OrchestrAI do next, and which teams must be involved?',
    scenario_id="b5",
)

QUESTION:
A customer renewal is at risk because support cases are recurring, a requested feature is still unavailable, and replacement hardware is delayed. What should OrchestrAI do next, and which teams must be involved?

RETRIEVED DOCUMENTS: 10
RETRIEVAL TIME (s): 0.0263

[1] Source: TEAMQA-LOG-001_OrchestrAI_Logistics_and_Field_Fulfillment_Handbook.docx | Department: Q&A Hanbooks
    exception decisions were made. Auditability is a first-class requirement because OrchestrAI often needs to reconstruct site hardware history months after deployment. How should warehouse stock counts be corrected after a mismatch? A recount must occur before any system adjustment is posted. If the mismatch remains, the warehouse lead documents the discrepancy, checks recent pick faces, receiving r...

[2] Source: VAL-CS-004_Renewal_Risk_Signals_Brief.docx | Department: Customer Support
 Effective date,2026-02-10,Review date,2026-08-10 Purpose: Defines the most important signals used by OrchestrAI Custom